# Track 2c: Sharper event information (Eshrit)

Builds on `track2_regimes.ipynb`. Its best model, `track2_gmm_probs_v1` (dev log RMSE 0.45160), got most of its
gain from three market-wide index-event flags. The HMM (`track2_hmm.ipynb`) added nothing. So this notebook
makes the event information sharper instead of making the regime model fancier.

Every variant is **the `track2_gmm_probs_v1` model plus one idea**, so each gain is measured against that model.

| Variant | Adds | Hypothesis |
|---|---|---|
| `track2_exposure_v1` | Stock-level passive ownership (ETF + index-fund shares as % of shares outstanding, and in days of volume), interacted with each event flag; S&P add/delete days | Index flows hit stocks in proportion to how much index money holds them. A market-wide flag treats all stocks alike, which is probably why Russell day got worse. |
| `track2_error_momentum_v1` | Yesterday's market-wide average error of the daily forecast (log actual − log forecast), and its 5-day mean | If the daily model under-forecasts the whole market for several days in a row, that is a regime defined directly by the errors we care about. |
| `track2_event_window_v1` | Indicators for 1 day and 2–5 days before and after each MSCI, Russell and S&P quarterly rebalance | Traders build up positions before an event and unwind after, so volume shifts around the event day, not only on it. |
| `track2_events_combined_v1` | All three | Do the gains add up? |

**Leakage controls.**
- Ownership comes from quarterly filings, used only from 60 days after the quarter ends (built into the file).
  Shares outstanding are taken from the previous trading day.
- Market-wide error uses the daily forecasts and auctions up to *yesterday's* close, from the full unsampled panel.
- The event calendar (MSCI, Russell, S&P quarterly) is published well in advance. S&P add/delete days are
  announced about a week before they take effect.
- The GMM is fitted only on earlier dates, as in track 2. K and the ridge strength are chosen on `fold.valid`.

In [1]:
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard, RIDGE_GRID, REBALANCE, SEED, OUT, INCUMBENT, ROOT
from notebooks.modeling_utils import fit_ridge, predict_ridge

OWNER = "Eshrit"
QUICK = False                     # True = 2024 only, not recorded
K_GRID = (2, 3, 4)
h = Harness(quick=QUICK)

## 1. Base: the track 2 GMM regime features (identical code to `track2_regimes.ipynb`)

In [2]:
cols = ["date", "permno", "cp_available", "cp_volume_surprise20", "cp_trade_count_surprise20",
        "cp_recent_30m_volume_share", "cp_vwap_return_prior_close"]
panel = h.full_panel("2021-01-01", "2026-01-01", cols)
live = panel[panel.cp_available.astype(bool)].copy()
live["log_surprise"] = np.log(live.cp_volume_surprise20.clip(lower=1e-3))
live["log_trade_surprise"] = np.log(live.cp_trade_count_surprise20.clip(lower=1e-3))
live["abs_move"] = live.cp_vwap_return_prior_close.abs()
live["hot"] = live.log_surprise > np.log(1.5)
g = live.groupby("date")
state = pd.DataFrame({
    "mkt_pace": g.log_surprise.median(),
    "mkt_trade_pace": g.log_trade_surprise.median(),
    "mkt_dispersion": g.log_surprise.quantile(0.75) - g.log_surprise.quantile(0.25),
    "mkt_breadth": g.hot.mean(),
    "mkt_recent30": g.cp_recent_30m_volume_share.median(),
    "mkt_abs_move": g.abs_move.median(),
})
STATE_COLS = list(state.columns)

@lru_cache(maxsize=None)
def regimes(train_end: str, k: int) -> pd.DataFrame:
    train = state.loc[state.index < pd.Timestamp(train_end), STATE_COLS]
    mu, sd = train.mean(), train.std()
    gmm = GaussianMixture(k, covariance_type="full", reg_covar=1e-3, n_init=5, random_state=SEED)
    gmm.fit(((train - mu) / sd).to_numpy())
    p = gmm.predict_proba(((state[STATE_COLS] - mu) / sd).to_numpy())
    order = np.argsort(gmm.means_[:, STATE_COLS.index("mkt_pace")])
    return pd.DataFrame(p[:, order], index=state.index, columns=[f"regime{k}_p{j}" for j in range(k)])

def valid_start(year):
    return "2021-10-01" if year == 2022 else f"{year - 1}-01-01"

## 2. New date-level features: event calendar, event windows, market-wide error momentum

In [3]:
EVENTS = {"russell": "russell_recon", "msci": "msci_review", "sp": "sp_quarterly_rebal"}
cal = h.con.sql(f'''
    SELECT CAST(date AS DATE) AS date,
           max(russell_recon) AS mkt_russell_recon, max(msci_review) AS mkt_msci_review, max(sp_quarterly_rebal) AS mkt_sp_rebal,
           min(td_to_next_russell_recon) AS to_russell, min(td_since_last_russell_recon) AS since_russell,
           min(td_to_next_msci_review) AS to_msci, min(td_since_last_msci_review) AS since_msci,
           min(td_to_next_sp_quarterly_rebal) AS to_sp, min(td_since_last_sp_quarterly_rebal) AS since_sp
    FROM read_parquet('{REBALANCE}') GROUP BY 1''').df()
cal["date"] = pd.to_datetime(cal.date)
cal = cal.set_index("date").sort_index()
EVENT_COLS = ["mkt_russell_recon", "mkt_msci_review", "mkt_sp_rebal"]

window = pd.DataFrame(index=cal.index)
for e in EVENTS:
    window[f"{e}_pre1"] = (cal[f"to_{e}"] == 1).astype(float)
    window[f"{e}_pre2_5"] = cal[f"to_{e}"].between(2, 5).astype(float)
    window[f"{e}_post1"] = (cal[f"since_{e}"] == 1).astype(float)
    window[f"{e}_post2_5"] = cal[f"since_{e}"].between(2, 5).astype(float)
WINDOW_COLS = list(window.columns)

# Market-wide error of the daily forecast, by date, then shifted so date t sees only dates up to t-1.
err = h.full_panel("2021-01-01", "2026-01-01", ["date", "log1p_auction_volume", "tree"])
err["e"] = err.log1p_auction_volume - np.log1p(err.tree)
daily_err = err.groupby("date").e.mean()
momentum = pd.DataFrame({"mkt_err_lag1": daily_err.shift(1),
                         "mkt_err_lag5": daily_err.shift(1).rolling(5, min_periods=3).mean()})
MOMENTUM_COLS = list(momentum.columns)

date_features = cal[EVENT_COLS].join(window).join(momentum, how="left")
print("Event-window days per year (MSCI 1 day before):", window.msci_pre1.groupby(window.index.year).sum().loc[2021:].to_dict())
date_features.loc["2021":, MOMENTUM_COLS].describe().round(4)

Event-window days per year (MSCI 1 day before): {2021: 4.0, 2022: 4.0, 2023: 4.0, 2024: 4.0, 2025: 4.0}


,mkt_err_lag1,mkt_err_lag5
count,1244.0000,1242.0000
mean,0.0119,0.0119
std,0.1433,0.0669
min,-0.8444,-0.1973
25%,-0.0748,-0.0329
50%,0.0022,0.0090
75%,0.0882,0.0544
max,0.9005,0.2594


## 3. New stock-level features: passive (index) ownership

Passive shares = ETF shares + index mutual-fund shares, from quarterly filings already lagged 60 days.
`passive_pct` divides by the previous trading day's shares outstanding. `log_passive_days` divides by the stock's
20-day average daily volume: how many days of normal trading the index money represents.

In [4]:
crsp = f"{ROOT}/blackrock_data/stock_daily/crsp_daily/*/*.parquet"
own = h.con.sql(f'''
    WITH so AS (
        SELECT permno, CAST(date AS DATE) AS date,
               lag(shares_outstanding_thousands) OVER (PARTITION BY permno ORDER BY date) * 1000.0 AS shares_out_lag1
        FROM read_parquet('{crsp}', union_by_name=true)
        WHERE date >= DATE '2020-11-01'
    )
    SELECT CAST(r.date AS DATE) AS date, r.permno,
           coalesce(r.etf_shares, 0) + coalesce(r.index_mf_shares, 0) AS passive_shares,
           (coalesce(r.etf_shares, 0) + coalesce(r.index_mf_shares, 0)) / nullif(so.shares_out_lag1, 0) AS passive_pct,
           r.etf_shares IS NULL AS own_missing,
           coalesce(r.sp500_add_day, 0) AS sp500_add_day, coalesce(r.sp500_del_day, 0) AS sp500_del_day,
           coalesce(r.sp500_member_x_sp_rebal, 0) AS sp500_member_x_sp_rebal
    FROM read_parquet('{REBALANCE}') r LEFT JOIN so ON so.permno = r.permno AND so.date = CAST(r.date AS DATE)
    WHERE r.date >= DATE '2021-01-01' ''').df()
own["date"] = pd.to_datetime(own.date)
own.loc[own.own_missing, ["passive_shares", "passive_pct"]] = np.nan
own["passive_pct"] = own.passive_pct.clip(0, 1)
own = own.drop(columns="own_missing").set_index(["date", "permno"])
own.passive_pct.describe().round(3)

count    1614767.000
mean           0.192
std            0.103
min            0.000
25%            0.128
50%            0.196
75%            0.257
max            1.000
Name: passive_pct, dtype: float64

In [5]:
def event_features(frame, fold_year, role):
    train_end = valid_start(fold_year) if role in ("tune", "valid") else f"{fold_year}-01-01"
    out = date_features.reindex(frame.date).set_axis(frame.index)
    out[EVENT_COLS] = out[EVENT_COLS].fillna(0)
    for k in K_GRID:
        out = out.join(regimes(train_end, k).reindex(frame.date).set_axis(frame.index))
    stock = own.reindex(pd.MultiIndex.from_frame(frame[["date", "permno"]])).set_axis(frame.index)
    out["passive_pct"] = stock.passive_pct
    out["log_passive_days"] = np.log1p(stock.passive_shares / frame.avg20_daily_volume.where(frame.avg20_daily_volume > 0))
    for col in ("sp500_add_day", "sp500_del_day", "sp500_member_x_sp_rebal"):
        out[col] = stock[col].fillna(0)
    for flag in EVENT_COLS:
        out[f"{flag}_x_passive_pct"] = out[flag] * out.passive_pct
        out[f"{flag}_x_passive_days"] = out[flag] * out.log_passive_days
    return out

h.add_features(event_features)
EXPOSURE_COLS = (["passive_pct", "log_passive_days", "sp500_add_day", "sp500_del_day", "sp500_member_x_sp_rebal"]
                 + [f"{f}_x_passive_pct" for f in EVENT_COLS] + [f"{f}_x_passive_days" for f in EVENT_COLS])

def base(k): return [f"regime{k}_p{j}" for j in range(k)] + EVENT_COLS
ADDS = {
    "track2_exposure_v1": EXPOSURE_COLS,
    "track2_error_momentum_v1": MOMENTUM_COLS,
    "track2_event_window_v1": WINDOW_COLS,
    "track2_events_combined_v1": EXPOSURE_COLS + MOMENTUM_COLS + WINDOW_COLS,
}
VARIANTS = {name: {k: base(k) + extra for k in K_GRID} for name, extra in ADDS.items()}
DESCRIPTIONS = {
    "track2_exposure_v1": "track2_gmm_probs_v1 + stock passive ownership (pct and days of volume) x event flags + S&P add/delete days",
    "track2_error_momentum_v1": "track2_gmm_probs_v1 + yesterday's market-wide daily-forecast error (lag 1 and 5-day mean)",
    "track2_event_window_v1": "track2_gmm_probs_v1 + 1 and 2-5 day before/after windows for MSCI, Russell, S&P quarterly",
    "track2_events_combined_v1": "track2_gmm_probs_v1 + passive exposure + error momentum + event windows",
}

## 4. Fit, tune and submit (K and ridge strength chosen on validation)

In [6]:
def tune_and_fit(fold, options):
    z_tune, bounds = fold.target(fold.tune), fold.bounds(fold.tune)
    y_valid = fold.valid.log1p_auction_volume.to_numpy(float)
    best = None
    for k, extra in options.items():
        prep = fold.preprocessor(fold.tune, extra_continuous=extra)
        x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
        for strength in RIDGE_GRID:
            beta = fit_ridge(x_tune, z_tune, strength)
            forecast = fold.to_shares(predict_ridge(x_valid, beta), fold.valid, bounds)
            rmse = float(np.sqrt(np.mean((np.log1p(forecast) - y_valid) ** 2)))
            if best is None or rmse < best[0]:
                best = (rmse, k, strength)
    rmse, k, strength = best
    prep = fold.preprocessor(fold.refit, extra_continuous=options[k])
    beta = fit_ridge(prep.transform(fold.refit), fold.target(fold.refit), strength)
    return predict_ridge(prep.transform(fold.test), beta), {"k": k, "strength": strength, "valid_rmse": rmse}

chosen = []
for fold in h.folds():
    for name, options in VARIANTS.items():
        correction, settings = tune_and_fit(fold, options)
        h.submit(fold, correction, name=name, owner=OWNER, description=DESCRIPTIONS[name])
        chosen.append({"year": fold.year, "variant": name, **settings})
pd.DataFrame(chosen)

track2_exposure_v1 2022: log RMSE 0.49430, WAPE 35.0735%, bias -14.79%


track2_error_momentum_v1 2022: log RMSE 0.49612, WAPE 35.0772%, bias -16.47%


track2_event_window_v1 2022: log RMSE 0.49492, WAPE 35.2873%, bias -13.95%


track2_events_combined_v1 2022: log RMSE 0.49776, WAPE 35.3256%, bias -16.24%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

track2_exposure_v1 2023: log RMSE 0.44360, WAPE 33.3476%, bias -12.64%


... (11 outputs left out)

track2_error_momentum_v1 2025: log RMSE 0.40468, WAPE 31.5115%, bias -5.25%


track2_event_window_v1 2025: log RMSE 0.40494, WAPE 31.4437%, bias -5.73%


track2_events_combined_v1 2025: log RMSE 0.40537, WAPE 31.6054%, bias -5.30%


,year,variant,k,strength,valid_rmse
0,2022,track2_exposure_v1,4,0.0100,0.552992
1,2022,track2_error_momentum_v1,4,0.0100,0.552829
2,2022,track2_event_window_v1,4,0.0010,0.552664
3,2022,track2_events_combined_v1,4,0.0100,0.550936
4,2023,track2_exposure_v1,2,0.0100,0.494305
5,2023,track2_error_momentum_v1,2,0.0100,0.495748
6,2023,track2_event_window_v1,2,0.0100,0.494281
7,2023,track2_events_combined_v1,2,0.0010,0.497226
8,2024,track2_exposure_v1,3,0.0100,0.443966
9,2024,track2_error_momentum_v1,3,0.0100,0.444358


## 5. Results

In [7]:
results = {name: h.report(name) for name in VARIANTS}
pd.concat({n: r["summary"].loc[[INCUMBENT, n]] for n, r in results.items()}).round(5)

development_rmse_log  \
                          model                                             
track2_exposure_v1        incumbent_tree_ridge                    0.45445   
                          track2_exposure_v1                      0.45160   
track2_error_momentum_v1  incumbent_tree_ridge                    0.45445   
                          track2_error_momentum_v1                0.45254   
track2_event_window_v1    incumbent_tree_ridge                    0.45445   
                          track2_event_window_v1                  0.45269   
track2_events_combined_v1 incumbent_tree_ridge                    0.45445   
                          track2_events_combined_v1               0.45342   

                                                     development_wape  \
                          model                                         
track2_exposure_v1        incumbent_tree_ridge                0.33838   
                          track2_exposure_v1                  0.33596   
track2_error_momentum_v1  incumbent_tree_ridge                0.33838   
                          track2_error_momentum_v1            0.33655   
track2_event_window_v1    incumbent_tree_ridge                0.33838   
                          track2_event_window_v1              0.33793   
track2_events_combined_v1 incumbent_tree_ridge                0.33838   
                          track2_events_combined_v1           0.33792   

                                                     development_bias  \
                          model                                         
track2_exposure_v1        incumbent_tree_ridge               -0.13937   
                          track2_exposure_v1                 -0.13537   
track2_error_momentum_v1  incumbent_tree_ridge               -0.13937   
                          track2_error_momentum_v1           -0.14068   
track2_event_window_v1    incumbent_tree_ridge               -0.13937   
                          track2_event_window_v1             -0.13130   
track2_events_combined_v1 incumbent_tree_ridge               -0.13937   
                          track2_events_combined_v1          -0.13978   

                                                     confirmation_rmse_log  \
                          model                                              
track2_exposure_v1        incumbent_tree_ridge                     0.40613   
                          track2_exposure_v1                       0.40395   
track2_error_momentum_v1  incumbent_tree_ridge                     0.40613   
                          track2_error_momentum_v1                 0.40468   
track2_event_window_v1    incumbent_tree_ridge                     0.40613   
                          track2_event_window_v1                   0.40494   
track2_events_combined_v1 incumbent_tree_ridge                     0.40613   
                          track2_events_combined_v1                0.40537   

                                                     confirmation_wape  
                          model                                         
track2_exposure_v1        incumbent_tree_ridge                 0.31597  
                          track2_exposure_v1                   0.31446  
track2_error_momentum_v1  incumbent_tree_ridge                 0.31597  
                          track2_error_momentum_v1             0.31512  
track2_event_window_v1    incumbent_tree_ridge                 0.31597  
                          track2_event_window_v1               0.31444  
track2_events_combined_v1 incumbent_tree_ridge                 0.31597  
                          track2_events_combined_v1            0.31605

In [8]:
pd.DataFrame({n: r["gate"] for n, r in results.items()}) if not QUICK else "Gate is only applied on the full 2022-2025 run."

,track2_exposure_v1,track2_error_momentum_v1,track2_event_window_v1,track2_events_combined_v1
dev_rmse_gain,0.002854,0.001913,0.001761,0.001035
gain_ok,True,False,False,False
worst_year_change,-0.000883,0.000936,-0.000266,0.002576
annual_ok,True,True,True,False
wape_ok,True,True,True,True
interval_ok,True,False,False,False
passes_gate,True,False,False,False


### Against the base model (`track2_gmm_probs_v1`)

The question for each idea is whether it adds anything on top of the base. Paired block-bootstrap differences
(variant minus base; negative means the idea helps).

In [9]:
ref = "track2_gmm_probs_v1"
path = OUT / "submissions" / ref / "predictions.parquet"
reference = pd.read_parquet(path)[["date", "permno", "forecast"]].rename(columns={"forecast": "ref"})
reference = reference[reference.date.dt.year.isin(h.years)]
rows = []
for name in VARIANTS:
    cand = pd.concat(h._submissions[name])[["date", "permno", "auction_volume", "forecast"]].rename(columns={"forecast": "cand"})
    paired = cand.merge(reference, on=["date", "permno"], validate="one_to_one")
    rows.append(h._paired_intervals(paired, "ref", "cand").query("metric in ['rmse_log', 'wape']").assign(variant=name))
pd.concat(rows)[["variant", "period", "metric", "difference", "lower", "upper"]].round(5)

,variant,period,metric,difference,lower,upper
0,track2_exposure_v1,development,rmse_log,-0.00001,-0.00027,0.00027
1,track2_exposure_v1,development,wape,-0.00018,-0.00072,0.00043
4,track2_exposure_v1,confirmation,rmse_log,-0.00007,-0.00046,0.00034
5,track2_exposure_v1,confirmation,wape,0.00058,-0.00052,0.00175
0,track2_error_momentum_v1,development,rmse_log,0.00094,0.00034,0.00158
1,track2_error_momentum_v1,development,wape,0.00041,-0.00011,0.00096
4,track2_error_momentum_v1,confirmation,rmse_log,0.00066,0.00019,0.00114
5,track2_error_momentum_v1,confirmation,wape,0.00124,0.00061,0.00194
0,track2_event_window_v1,development,rmse_log,0.00109,0.00024,0.00195
1,track2_event_window_v1,development,wape,0.00179,0.00082,0.00270


Segment log RMSE (lower is better), with the base model for comparison.

In [10]:
h._submissions[ref] = [reference.merge(pd.concat(h._submissions[next(iter(VARIANTS))])[["date", "permno", "auction_volume", "cp_available", "anchor_forecast"]],
                                       on=["date", "permno"]).rename(columns={"ref": "forecast"})]
base_seg = h.report(ref, record=False)["segments"]
table = base_seg[["daily_anchor", INCUMBENT, ref]].join(pd.DataFrame({n: r["segments"][n] for n, r in results.items()}))
table.join(table[list(VARIANTS)].sub(table[ref], axis=0).add_suffix(" vs base")).round(4)

,daily_anchor,incumbent_tree_ridge,track2_gmm_probs_v1,track2_exposure_v1,track2_error_momentum_v1,track2_event_window_v1,track2_events_combined_v1,track2_exposure_v1 vs base,track2_error_momentum_v1 vs base,track2_event_window_v1 vs base,track2_events_combined_v1 vs base
segment,,,,,,,,,,,
all,0.4656,0.4561,0.4533,0.4534,0.4544,0.4544,0.4553,0.0000,0.0010,0.0011,0.0020
fomc_day,0.4957,0.4796,0.4754,0.4748,0.4752,0.4725,0.4704,-0.0006,-0.0002,-0.0029,-0.0050
largest_1pct_auctions (diagnostic),1.1105,1.0681,1.0552,1.0549,1.0606,1.0569,1.0641,-0.0004,0.0054,0.0017,0.0088
low_liquidity,0.5213,0.5234,0.5209,0.5206,0.5218,0.5218,0.5221,-0.0003,0.0009,0.0008,0.0011
month_end,0.6563,0.6526,0.6363,0.6364,0.6385,0.6359,0.6377,0.0001,0.0022,-0.0004,0.0014
msci_review,0.7786,0.7734,0.7149,0.7141,0.7131,0.7140,0.7121,-0.0008,-0.0017,-0.0009,-0.0027
quarterly_expiry,0.7610,0.7308,0.7314,0.7270,0.7305,0.7313,0.7264,-0.0044,-0.0009,-0.0002,-0.0050
russell_recon,1.0273,1.0112,1.0194,1.0233,1.0301,1.0283,1.0472,0.0039,0.0107,0.0088,0.0277
short_history,1.0171,0.9970,0.9956,0.9943,0.9981,0.9919,1.0008,-0.0013,0.0025,-0.0036,0.0052


In [11]:
leaderboard()[["name", "development_rmse_log", "development_wape", "confirmation_rmse_log", "dev_rmse_gain", "passes_gate"]]

,name,development_rmse_log,development_wape,confirmation_rmse_log,dev_rmse_gain,passes_gate
0,track2_exposure_v1,0.451597,0.335962,0.403949,0.002854,True
1,track2_gmm_interact_v1,0.451602,0.336080,0.404027,0.002850,True
2,track2_gmm_probs_v1,0.451603,0.336145,0.404022,0.002848,True
3,track2_hmm_live_v1,0.451856,0.336376,0.403953,0.002595,True
4,track2_events_only_v1,0.452110,0.336405,0.403971,0.002341,True
5,track2_hmm_slow_v1,0.452421,0.336382,0.403975,0.002030,True
6,track2_error_momentum_v1,0.452538,0.336552,0.404680,0.001913,False
7,track2_event_window_v1,0.452690,0.337931,0.404938,0.001761,False
8,track2_market_state_v1,0.452958,0.339121,0.405135,0.001493,False
9,track2_events_combined_v1,0.453416,0.337916,0.405366,0.001035,False


## Findings (full run, 2022–2025)

| Variant | Dev log RMSE | Change vs base (95% interval) | 2025 log RMSE | Gate vs incumbent |
|---|---|---|---|---|
| Base: `track2_gmm_probs_v1` | 0.45160 | | 0.40402 | pass |
| `track2_exposure_v1` | 0.45160 | −0.00001 (−0.00027 to +0.00027) | 0.40395 | pass |
| `track2_error_momentum_v1` | 0.45254 | +0.00094 (+0.00034 to +0.00158) | 0.40468 | fail |
| `track2_event_window_v1` | 0.45269 | +0.00109 (+0.00024 to +0.00195) | 0.40494 | fail |
| `track2_events_combined_v1` | 0.45342 | +0.00181 (+0.00064 to +0.00298) | 0.40537 | fail |

1. **None of the three ideas improves on the base.**
   - Passive-ownership exposure is a tie overall. It helps quarterly-expiry days slightly (−0.004), but Russell day still gets worse.
   - Error momentum and event windows make things *worse*, and their intervals lie entirely above zero.
   - Combining them is worse still.
2. **Likely reason: too few training event days.** The 15:30 checkpoint panel starts in 2021. By the 2024
   fold, the model has seen only 3 Russell days, about 12 MSCI days and 12 S&P quarterly rebalances. Every new
   event-related coefficient is learned from a handful of days, so it fits noise. The combined variant shows this
   directly: it had the best validation score in 2022 (0.5509) and then the worst test scores.
3. **Market-wide error momentum is mostly noise.** Yesterday's average error swings a lot from day to day
   (std 0.14), and the daily model already uses the stock's own recent auctions.

**Verdict:** Track 2's candidate stays `track2_gmm_probs_v1`. Further event features need either more history
(an intraday panel before 2021, if the data exists) or pooling across event types, not more columns.
